<a href="https://colab.research.google.com/github/sechali/buku_ajar_ai/blob/main/notebooks/part-05/AI_Modul_5.1_Praktikum_Konsep_Dasar_Machine_Learning.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Praktikum AI Modul 5.1: Konsep Dasar Machine Learning
Program Studi Sarjana Sains Data / Kecerdasan Buatan / Teknik Pertanian  
Fakultas Teknologi Pertanian, Institut Pertanian STIPER Yogyakarta

---

### Capaian Pembelajaran Praktikum:
1. Mengamati perbandingan empiris antara sistem berbasis aturan manual (*Rule-Based*) dengan sistem pembelajaran mesin (*Machine Learning*).
2. Memverifikasi kerangka formal Tom Mitchell: membuktikan secara empiris bahwa penambahan pengalaman ($E$) meningkatkan performa model ($P$) pada suatu tugas ($T$) melalui Kurva Pembelajaran (*Learning Curve*).
3. Memvisualisasikan aproksimasi fungsi hipotesis $\hat{h}(x)$ dalam mendekati fungsi sejati alamiah $f(x)$.
4. Mendemonstrasikan Teorema *No Free Lunch* (NFL) melalui eksperimen komparasi model linier vs pohon keputusan.

In [ ]:
import os
os.environ['MPLBACKEND'] = 'Agg'
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, learning_curve
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.tree import DecisionTreeRegressor, DecisionTreeClassifier
from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics import accuracy_score, mean_squared_error, r2_score

print(f"NumPy version: {np.__version__}")
print(f"Pandas version: {pd.__version__}")


## 1. Komparasi Paradigma: Rule-Based vs Machine Learning pada Penentuan Mutu CPO
Kita akan membandingkan sistem berbasis aturan manual (if-else kaku) dengan model *Machine Learning* dalam mengklasifikasikan mutu minyak sawit mentah (*Crude Palm Oil* - CPO) berdasarkan dua parameter laboratorium: Kadar Asam Lemak Bebas (FFA, %) dan Kadar Air (Moisture, %).


In [ ]:
# 1. Bangun dataset simulasi pengujian mutu CPO pabrik
np.random.seed(42)
n_cpo = 300

ffa = np.random.uniform(1.5, 7.0, n_cpo)
kadar_air = np.random.uniform(0.05, 0.40, n_cpo)

# Fungsi alamiah sejati (ground truth dengan noise lapangan)
# Mutu Prima (1) jika kombinasi FFA dan Air berada di bawah batas non-linier
skor_kualitas = 0.6 * ffa + 10.0 * kadar_air + np.random.normal(0, 0.25, n_cpo)
label_prima = (skor_kualitas < 4.2).astype(int) # 1 = Prima, 0 = Standar/Afkir

df_cpo = pd.DataFrame({
    'Kadar_FFA': ffa.round(2),
    'Kadar_Air': kadar_air.round(3),
    'Mutu_Prima': label_prima
})

# Pisahkan data latih dan uji
X = df_cpo[['Kadar_FFA', 'Kadar_Air']]
y = df_cpo['Mutu_Prima']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# Pendekatan 1: Rule-Based Manual (Aturan Kaku Buatan Analis)
def sistem_aturan_manual(data):
    prediksi = []
    for _, row in data.iterrows():
        # Aturan kaku yang tidak mempertimbangkan interaksi kontinyu
        if row['Kadar_FFA'] <= 3.8 and row['Kadar_Air'] <= 0.18:
            prediksi.append(1)
        else:
            prediksi.append(0)
    return np.array(prediksi)

y_pred_rule = sistem_aturan_manual(X_test)
acc_rule = accuracy_score(y_test, y_pred_rule)

# Pendekatan 2: Machine Learning (Logistic Regression mempelajari batas dari data latih)
model_ml = LogisticRegression()
model_ml.fit(X_train, y_train)
y_pred_ml = model_ml.predict(X_test)
acc_ml = accuracy_score(y_test, y_pred_ml)

print(f"Akurasi Sistem Aturan Manual (Rule-Based): {acc_rule * 100:.2f}%")
print(f"Akurasi Model Machine Learning:            {acc_ml * 100:.2f}%")
print(f"Peningkatan Akurasi Berbasis Data:          {(acc_ml - acc_rule) * 100:+.2f}%")


## 2. Verifikasi Empiris Definisi Tom Mitchell ($E, T, P$)
Sesuai definisi Mitchell:
> *Sebuah program komputer dikatakan belajar jika performanya ($P$) pada tugas ($T$) meningkat seiring bertambahnya pengalaman ($E$).*

Kita akan melacak metrik error ($P = \text{RMSE}$) saat ukuran dataset pengalaman ($E$) dinaikkan secara bertahap dari 20 sampel hingga 400 sampel pada tugas prediksi tonase panen ($T$).


In [ ]:
# 1. Dataset pengalaman agronomis: estimasi tonase panen TBS
n_samples_total = 500
np.random.seed(10)

curah_hujan = np.random.uniform(100, 350, n_samples_total)
pupuk = np.random.uniform(150, 450, n_samples_total)
# Target fisik sejati
tonase_aktual = 0.03 * curah_hujan + 0.02 * pupuk + np.random.normal(0, 1.2, n_samples_total)

X_kebun = np.column_stack([curah_hujan, pupuk])
y_kebun = tonase_aktual

# Ukuran pengalaman (Experience E) bervariasi
ukuran_pengalaman = [20, 50, 100, 200, 350]
daftar_rmse_uji = []

# Evaluasi pada data uji tetap (Holdout Test Set)
X_train_full, X_test_holdout, y_train_full, y_test_holdout = train_test_split(
    X_kebun, y_kebun, test_size=100, random_state=42
)

for n_e in ukuran_pengalaman:
    # Ambil subset data pengalaman sebesar n_e
    X_sub = X_train_full[:n_e]
    y_sub = y_train_full[:n_e]
    
    # Latih model
    reg = LinearRegression()
    reg.fit(X_sub, y_sub)
    
    # Ukur performa P pada data uji
    y_pred_holdout = reg.predict(X_test_holdout)
    rmse = np.sqrt(mean_squared_error(y_test_holdout, y_pred_holdout))
    daftar_rmse_uji.append(rmse)

df_mitchell = pd.DataFrame({
    'Pengalaman_E (Jumlah Sampel)': ukuran_pengalaman,
    'Performa_P (RMSE Galat)': np.round(daftar_rmse_uji, 4)
})
print("=== BUKTI EMPIRIS FORMULASI TOM MITCHELL ===")
print(df_mitchell.to_string(index=False))
print("\nKesimpulan: Performa model terbukti membaik (galat RMSE menurun) seiring penambahan pengalaman data!")


## 3. Visualisasi Pendekatan Fungsi Hipotesis $\hat{h}(x)$ terhadap Fungsi Sejati $f(x)$
Dalam sains data perkebunan, fungsi alamiah $f(x)$ yang menghubungkan dosis pupuk dengan respons hasil tanaman tidak pernah diketahui secara pasti. Model kecerdasan buatan mencari fungsi aproksimasi $\hat{h}(x)$ terbaik dari ruang hipotesisnya.


In [ ]:
# Fungsi sejati alamiah (non-linier kuadratik respons tanaman - Hukum Hasil yang Menurun)
x_pupuk = np.linspace(50, 400, 60)
def f_sejati(x):
    return -0.00015 * (x - 250)**2 + 25.0

y_sejati = f_sejati(x_pupuk)
y_observasi = y_sejati + np.random.normal(0, 1.2, len(x_pupuk))

# Model 1: Hipotesis Linier Derajat 1 (Underfitting)
h_linier = LinearRegression()
h_linier.fit(x_pupuk.reshape(-1, 1), y_observasi)
pred_linier = h_linier.predict(x_pupuk.reshape(-1, 1))

# Model 2: Hipotesis Pohon Keputusan (Non-Parametrik Segmental)
h_tree = DecisionTreeRegressor(max_depth=3)
h_tree.fit(x_pupuk.reshape(-1, 1), y_observasi)
pred_tree = h_tree.predict(x_pupuk.reshape(-1, 1))

print("Model berhasil dilatih untuk mendekati fungsi sejati tanaman.")
print(f"R2 Hipotesis Linier:          {r2_score(y_observasi, pred_linier):.4f}")
print(f"R2 Hipotesis Decision Tree:   {r2_score(y_observasi, pred_tree):.4f}")


## 4. Demonstrasi Teorema No Free Lunch (NFL)
Teorema *No Free Lunch* menyatakan bahwa tidak ada algoritma tunggal yang selalu mengalahkan algoritma lainnya di seluruh ragam masalah. Kita buktikan dengan menguji **Model Linier** vs **K-Nearest Neighbors (KNN)** pada dua kasus tanah yang berbeda secara drastis:
- **Kasus 1:** Hubungan fisik murni linier (e.g. kedalaman olah tanah vs infiltrasi air).
- **Kasus 2:** Hubungan fisik sangat non-linier bergelombang (e.g. fluktuasi kelembaban tanah diurnal).


In [ ]:
# Dataset Kasus 1: Hubungan Linier
x_lin = np.random.uniform(10, 100, 200).reshape(-1, 1)
y_lin = 3.5 * x_lin.ravel() + 12.0 + np.random.normal(0, 8.0, 200)

# Dataset Kasus 2: Hubungan Non-Linier Bergelombang
x_non = np.random.uniform(0, 10, 200).reshape(-1, 1)
y_non = np.sin(x_non.ravel() * 1.5) * 20.0 + np.random.normal(0, 2.0, 200)

# Uji kedua algoritma pada Kasus 1 (Linier)
X1_tr, X1_te, y1_tr, y1_te = train_test_split(x_lin, y_lin, test_size=0.3, random_state=42)
m_lin_1 = LinearRegression().fit(X1_tr, y1_tr)
m_knn_1 = KNeighborsRegressor(n_neighbors=5).fit(X1_tr, y1_tr)

r2_lin_1 = r2_score(y1_te, m_lin_1.predict(X1_te))
r2_knn_1 = r2_score(y1_te, m_knn_1.predict(X1_te))

# Uji kedua algoritma pada Kasus 2 (Non-Linier)
X2_tr, X2_te, y2_tr, y2_te = train_test_split(x_non, y_non, test_size=0.3, random_state=42)
m_lin_2 = LinearRegression().fit(X2_tr, y2_tr)
m_knn_2 = KNeighborsRegressor(n_neighbors=5).fit(X2_tr, y2_tr)

r2_lin_2 = r2_score(y2_te, m_lin_2.predict(X2_te))
r2_knn_2 = r2_score(y2_te, m_knn_2.predict(X2_te))

print("=== BUKTI TEOREMA NO FREE LUNCH (NFL) ===")
print("1. Kasus Karakteristik Data Linier:")
print(f"   - Linear Regression R2: {r2_lin_1:.4f}  <-- Pemenang!")
print(f"   - KNN Regressor     R2: {r2_knn_1:.4f}")
print("\n2. Kasus Karakteristik Data Non-Linier Bergelombang:")
print(f"   - Linear Regression R2: {r2_lin_2:.4f}  <-- Gagal total!")
print(f"   - KNN Regressor     R2: {r2_knn_2:.4f}  <-- Pemenang!")
print("\nKesimpulan: Algoritma terbaik ditentukan oleh kesesuaian asumsi (bias induktif) dengan pola data!")


## 5. Tugas Tantangan Eksplorasi Mandiri (HOTS Challenge)

### Tantangan: Evaluasi Bias Induktif pada Deteksi Anomali Sensor Suhu
Sebuah sensor IoT di tangki perebusan kelapa sawit (*sterilizer*) menghasilkan data kontinu. Jika Anda hanya memiliki 30 sampel data suhu, ujilah apakah model berkapasitas parameter rendah (Regresi Linier) atau model non-parametrik kompleks (Pohon Keputusan tanpa batasan kedalaman) yang menghasilkan error generalisasi lebih rendah!

### Solusi Tantangan:


In [ ]:
# Simulasi 30 sampel data kecil (Small-data regime)
np.random.seed(99)
x_sensor = np.sort(np.random.uniform(70, 140, 30)).reshape(-1, 1)
# Hubungan fisik: Tekanan uap sebanding dengan kuadrat suhu
y_tekanan = 0.005 * (x_sensor.ravel() ** 1.8) + np.random.normal(0, 0.5, 30)

x_tr, x_te, y_tr, y_te = train_test_split(x_sensor, y_tekanan, test_size=10, random_state=42)

# Model Sederhana: Linear Regression (Bias kuat, varians rendah)
mod_simple = LinearRegression()
mod_simple.fit(x_tr, y_tr)
mse_simple = mean_squared_error(y_te, mod_simple.predict(x_te))

# Model Kompleks: Decision Tree tanpa batas kedalaman (Bias rendah, varians sangat tinggi)
mod_complex = DecisionTreeRegressor(max_depth=None, random_state=42)
mod_complex.fit(x_tr, y_tr)
mse_complex = mean_squared_error(y_te, mod_complex.predict(x_te))

print("=== HASIL EVALUASI BIAS INDUKTIF PADA DATA KECIL ===")
print(f"MSE Model Sederhana (Linear Regression): {mse_simple:.4f}")
print(f"MSE Model Kompleks (Unconstrained Tree):  {mse_complex:.4f}")
print(f"Rasio Kesalahan: Model Kompleks memiliki error {mse_complex/mse_simple:.1f}x lebih besar karena overfitting!")
print("Rekomendasi Occam's Razor: Pada data berukuran kecil, pilihlah model berkapasitas rendah dengan bias induktif sederhana.")
